In [ ]:
import json
from collections import OrderedDict
from pprint import pprint
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pyLDAvis
from tqdm import tqdm_notebook as tqdm 
import pyLDAvis.gensim_models
from gensim import models, corpora

__Author-Topic Latent Dirichlet Allocation__: This notebook uses the corpus, dictionary and other data built in [Text_Mining_Processing.ipynb](Text_Mining_Processing.ipynb) to train a Author-Topic LDA model.

- Latent Dirichlet Allocation ([LDA](https://en.wikipedia.org/wiki/Dirichlet_distribution)) by [Blei, Ng, & Jordan](http://www.jmlr.org/papers/v3/blei03a.html) is a topic modelling technique. In a text corpus, each document is associated with a multinomial distribution over topics, and each topic is associated with a multinomial distribution over words. Moreover, Dirichlet distributions are used as prior distributions.

- The author topic model extends LDA to include authorship information.  For the author-topic model, each author is associated with a multonimial distribution over topics (instead of each documents). In addition, the model can also handle documents with more than one author.

- The models can be represented graphically taken from [Rosen Zvi, Griffiths, Steyvers, & Smyth](https://arxiv.org/abs/1207.4169).


- The algorithms provided by the [gensim](https://radimrehurek.com/gensim/) Python library, which are based on [Hoffman, Bach, & Blei](http://papers.nips.cc/paper/3902-online-learning-for-latentdirichlet-allocation!) for LDA and [Rosen-Zvi, Griffiths, Steyvers, & Smyth](https://arxiv.org/abs/1207.4169) for the author topic extension. 

-

In [ ]:
dictionary = corpora.Dictionary.load('data/dictionary')
corpus = list(corpora.MmCorpus('data/corpus.mm'))
with open('data/author2doc.json') as f: 
    author2doc = json.load(f)
with open('data/tokenized_data.json') as f: 
    tokenized_data = json.load(f)
    

__Topic coherence__: The number of topics needs to be set for LDA, and it is often a highly non-trivial task.

- The approach here is to train LDA models with different values of number of topics, and choose the one with the highest coherence value. 

- Choosing a value that marks the end of a rapid growth of topic coherence is a better method to obtain meaning full and interpretable topics.

- Typically, if the same words is repeated in multiple topics, it is a sign that the number of topics is too large. There are other approaches to evaluate the topic models such as perplexity. However, topic coherence is usually better correlated to human judgment than perplexity.

In [ ]:

def compute_coherence_values(dictionary, corpus, texts, author2doc, n_min, n_max, step=1, return_models=False):
    """
    Compute c_v coherence for different numbers of topics

    Args:
    ----
    dictionary: Gensim dictionary
    corpus: Gensim corpus
    texts (list of list of str): List of input tokenized texts.
    n_min (int): min number of topics.
    n_max (int): max number of topics.

    Returns:
    -------
    trained_models (OrderedDict): Dictionary of LDA topic models corresponding to a number of topics.
    coherence_values (OrderedDict): Dictionary of Coherence values corresponding to a number of topics.
    """
    trained_models = OrderedDict()
    coherence_values = OrderedDict()
    
    for n_topics in range(n_min, n_max, step):
        trained_models[n_topics] = models.AuthorTopicModel(corpus,
                                                           num_topics=n_topics,
                                                           id2word=dictionary,
                                                           author2doc=author2doc, 
                                                           passes=50,
                                                           alpha='auto',
                                                           eta='symmetric',
                                                           gamma_threshold=1e-5,
                                                           random_state=10)
        
    cm = models.CoherenceModel.for_models(trained_models.values(),
                                          dictionary=dictionary,
                                          texts=texts,
                                          coherence='c_v')
    
    coherence_estimates = cm.compare_models(trained_models.values())
    coherence_values = OrderedDict(zip(trained_models.keys(), [c for _, c in coherence_estimates]))
    if return_models:
        return coherence_values, trained_models
    else:
        return coherence_values


In [ ]:
def plot_coherence(coherences_dict, ax=None):
    """
    Plot coherence values as a function of the number of topics
    
    Args:
    ----
    coherences_dict (dict of int: float): dictionary of coherence values with num_topics as keys.
    ax (matplotlib Axes object)
    """
    if ax is None:
        ax = plt.gca()
    ntopics_vs_coherence = list(coherences_dict.items())
    ntopics_vs_coherence.sort()
    best_n, max_coherence = max(ntopics_vs_coherence, key=lambda x: x[1])
    ax.plot(*zip(*ntopics_vs_coherence))
    c = ax._get_lines.get_next_color()
    ax.axvline(best_n, linestyle='-.', color=c, label='coherence maxima')
    ax.set_xticks([n for n, _ in ntopics_vs_coherence])
    ax.set_xlabel('number of topics')
    ax.set_ylabel('coherence')
    ax.legend()

In [ ]:
coherence_values = compute_coherence_values(dictionary, corpus, tokenized_data, author2doc,
                                            n_min=2, n_max=15)

In [ ]:
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-notebook")
plot_coherence(coherence_values)
plt.title("c_v coherence")
plt.show()

In the above the shape of the curve depends quite a lot on the random seed.  After trying out several numbers and random seeds, it is found that a higher number of topics is good to detect more unusual and sparsely represented topics.  

__Training__:  It is generally a good a idea to choose an asymmetric Dirichlet prior over the document-topic distributions (or author-topic), and a symetric one over the topic-word distributions. Those correspond to `alpha` and `eta` in the gensim implementation of LDA, respectively. Additionally, `alpha='auto'`,  is used so that the asymmetric prior is learnt directly from the data.

Also a low value is use for `gamma_threshold`, which is the threshold value of $\gamma$ (the topic difference between consecutive two topics, i.e. variations between subsequent inferences) until which the iterations continue.

In [ ]:
num_topics = 8

lda_atmodel = models.AuthorTopicModel(
    corpus,
    num_topics=num_topics,
    id2word=dictionary,     # dict of int: str (word id -> word)
    author2doc=author2doc,  # dict of str: int (author -> document id)
    passes=50,              # number of passes through the corpus during training
    alpha='auto',           # parameter of the Dirichlet prior on the per-document topic distributions
    eta='symmetric',        # parameter of the Dirichlet prior on the per-topic word distributions 
    random_state=10,
    gamma_threshold=1e-5    # Convergence criterion of the `online variational Bayes for LDA` algorithm
)

print("The α's obtained from `alpha='auto'` are:")
print(lda_atmodel.alpha)

__Topic keywords__: The next step is to examine the obtained topics and the associated _keywords_. The most relevant keywords of each topics are printed below.

- A Python library for interactive topic model visualization is  included which is a  nice interactive chart of the pyLDAvis.

- As expected of SF novels, some themes are dominant (e.g. space travel). As a result, several topics share the same overall theme, and it is difficult to tell them apart. However, the overall theme is interpretable.

- It is however remarkable that the algorithm could detect some topics only sparsely represented in the corpus. This would not have been possible without an asymmetric `alpha` (e.g. topic 3 below, about family). There is also major topics (e.g. topic 4 below), which countain many important keywords. Those are themes appearing very often in the SF litterature (e.g. planets, power & control).

In [ ]:
print("Author-Topic LDA Model:")
# Print the Keyword in all topics
for (index, keywords) in lda_atmodel.print_topics(-1, 8):
    print(f'Topic #{index + 1}:')
    print('\t', keywords)
# pprint(lda_atmodel.print_topics(-1, 10))

__Topic visualization__: Simply using the top keywords with the highest weights is not always the best method to interpret topics.

- Using interactive chart of pyLDAvis, we can adjust the relevance metric $\lambda$, which is used to calculate the _relevance_ of the words. It appears  that $\lambda \approx 0.6$ gives interesting results and helps interepret the topics.

- The pyLDAvis/gensim.py file is modify in order to use the library with the author-topic model here line 48 is replaced by

- `gamma, _ = topic_model.inference(corpus)` with

- `doc2author = gensim.models.atmodel.construct_doc2author(topic_model.corpus, topic_model.author2doc)`<p>
`gamma, _ = topic_model.inference(topic_model.corpus, topic_model.author2doc, doc2author, 0)`

In [ ]:
from gensim.models import LdaModel

lda_vis_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=10,
    passes=20,
    random_state=10
)

In [ ]:
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

vis = gensimvis.prepare(
    lda_vis_model,
    corpus,
    dictionary
)

pyLDAvis.display(vis)

In [ ]:
for i, topic in lda_atmodel.print_topics(num_words=10):
    print(f"Topic {i}: {topic}")

In [ ]:
topics = lda_atmodel.show_topics(
    num_topics=-1,
    num_words=10,
    formatted=False
)

for t in topics:
    print(t)

In [ ]:
def plot_topic_distribution(author, num_topics, ax=None):
    if ax is None:
        ax = plt.gca()
        
    x = list(range(1, num_topics+1))
    height = np.zeros(num_topics)
    author_topics = lda_atmodel.get_author_topics(author)
    for i, h in author_topics:
        height[i] = h
    ax.bar(x=x, height=height, tick_label=x)
    n_novels = len(author2doc[author])
    n_tokens = sum(len(tokenized_data[i]) for i in author2doc[author])
    ax.set_title(f'{author} ({n_novels} novels, {n_tokens} tokens)')
    ax.set_ylim([0, 1])
    
def plot_topic_distribution_by_author(authors, num_topics, ax=None):
    n_authors = len(authors)
    n_col = 3
    n_row = (n_authors - 1) // n_col + 1
    size = 4.5
    fig, axes = plt.subplots(n_row, n_col, figsize=(size*n_col, size*0.8*n_row))
    axes = axes.flat
    for ax, author in zip(axes, subset_of_authors):
        plot_topic_distribution(author, num_topics, ax=ax)
    fig.suptitle('Topic distribution by author', y=1.1, fontsize=20)
    plt.tight_layout()

In [ ]:
subset_of_authors = ['Isaac Asimov', 'Frank Herbert', 'Philip K. Dick', 'Liu Cixin']
plot_topic_distribution_by_author(subset_of_authors, num_topics)

In the figure topic 4 appears for all 4 authors, looking at Isaac Asimov, topics 4 & 6 are about planets, power, and colonization. Frank Hebert's Dune is all about force and power in a interplanetary setup, topic 4 is very fitting.